# 02_16b Kaggle MedNeXt-S PHE-only stronger baseline

Notebook nay la ban manh hon cua `02_16`: van dung official `MIC-DKFZ/MedNeXt`, van PHE-only, van khong dung ICH teacher/prior, va van dung cung split 99/10/11 nhu `02_12b`.

Muc tieu:

```text
input  channel 0 = CT only
target label     = manual PHE mask
model            = official MIC-DKFZ MedNeXt-S
```

Khac voi `02_16`, ban nay them augmentation, tang so patch train moi epoch, dung focal+dice loss, tune threshold tren validation, va thu post-processing connected component tren validation truoc khi predict test.


In [1]:
from pathlib import Path
import os
import sys
import re
import json
import math
import time
import random
import gc
import subprocess

import numpy as np
import pandas as pd

IS_KAGGLE = Path('/kaggle/input').exists()
KAGGLE_INPUT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path.cwd().resolve()
LOCAL_ROOT = Path.cwd().resolve()

# Optional manual overrides. Neu auto-detect sai, gan path nay tren Kaggle.
# Example:
# USER_PHE_ROOT = Path('/kaggle/input/datasets/dovune/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT')
# USER_MEDNEXT_REPO = Path('/kaggle/input/mednext/MedNeXt')
USER_PHE_ROOT = None
USER_MEDNEXT_REPO = None
MEDNEXT_REPO_URL = 'https://github.com/MIC-DKFZ/MedNeXt.git'

OUTPUT_ROOT = WORK_ROOT / 'outputs_02_16b_kaggle_mednext_s_phe_only_stronger_baseline'
PRED_DIR = OUTPUT_ROOT / 'predictions_mednext_s_best_tuned'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
for p in [OUTPUT_ROOT, PRED_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
VAL_SCAN_IDS = {'0013', '0014', '0060', '0078', '0080', '0115', '0119', '0130', '0138', '0160'}
TEST_SCAN_IDS = {'0002', '0029', '0033', '0051', '0061', '0084', '0095', '0096', '0113', '0116', '0167'}

# Stronger than 02_16, still small enough for Kaggle T4.
MAX_EPOCHS = 160
TRAIN_PATCHES_PER_EPOCH = 256
PATCH_SIZE = (32, 128, 128)  # z, y, x. Neu OOM thi giam ve (32, 112, 112).
BATCH_SIZE = 1
FG_PATCH_PROB = 0.85
VALIDATE_EVERY = 10
NUM_WORKERS = 0

# Official MedNeXt-S config. model_id S la Small theo MIC-DKFZ/MedNeXt.
MEDNEXT_MODEL_ID = 'S'
MEDNEXT_KERNEL_SIZE = 3
MEDNEXT_DEEP_SUPERVISION = False

LR = 5e-4
WEIGHT_DECAY = 1e-5
USE_AMP = True
GRAD_CLIP_NORM = 12.0

# CT-only baseline. Khong dung ICH.
CT_WINDOW = (-100.0, 200.0)
THRESHOLD = 0.5
THRESHOLD_GRID = tuple(float(x) for x in np.round(np.arange(0.20, 0.76, 0.05), 2))
POSTPROCESS_MIN_COMPONENT_SIZE_GRID = (0, 10, 50, 100)
DEFAULT_POSTPROCESS_MIN_COMPONENT_SIZE = 0
SLIDING_WINDOW_OVERLAP = 0.5

# Lightweight 3D augmentation.
AUG_FLIP_PROB = 0.5
AUG_INTENSITY_SHIFT = 0.08
AUG_INTENSITY_SCALE = 0.12
AUG_GAUSSIAN_NOISE_STD = 0.03

print('IS_KAGGLE   :', IS_KAGGLE)
print('WORK_ROOT   :', WORK_ROOT)
print('OUTPUT_ROOT :', OUTPUT_ROOT)
print('MAX_EPOCHS  :', MAX_EPOCHS)
print('PATCH_SIZE  :', PATCH_SIZE)
print('PATCHES/EPOCH:', TRAIN_PATCHES_PER_EPOCH)
print('threshold grid:', THRESHOLD_GRID)
print('component grid:', POSTPROCESS_MIN_COMPONENT_SIZE_GRID)


IS_KAGGLE   : True
WORK_ROOT   : /kaggle/working
OUTPUT_ROOT : /kaggle/working/outputs_02_16b_kaggle_mednext_s_phe_only_stronger_baseline
MAX_EPOCHS  : 160
PATCH_SIZE  : (32, 128, 128)
PATCHES/EPOCH: 256
threshold grid: (0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75)
component grid: (0, 10, 50, 100)


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
    except Exception:
        pass

seed_everything(SEED)


def ensure_import(import_name, pip_name=None):
    import importlib
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        cmd = [sys.executable, '-m', 'pip', 'install', pip_name]
        print('Missing', import_name, '-> running:', ' '.join(cmd))
        subprocess.check_call(cmd)
        return importlib.import_module(import_name)

sitk = ensure_import('SimpleITK')
ensure_import('scipy')

import SimpleITK as sitk
from scipy import ndimage as ndi

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler



def has_mednext_repo(path_like):
    p = Path(path_like)
    return (p / 'nnunet_mednext').exists() and (p / 'setup.py').exists()


def find_mednext_repo():
    if USER_MEDNEXT_REPO is not None:
        p = Path(USER_MEDNEXT_REPO)
        if has_mednext_repo(p):
            return p
        raise FileNotFoundError(f'USER_MEDNEXT_REPO is not a MedNeXt repo: {p}')

    candidates = [LOCAL_ROOT / 'MedNeXt', LOCAL_ROOT / 'mednext']
    if IS_KAGGLE:
        for base in KAGGLE_INPUT.iterdir():
            candidates.extend([base, base / 'MedNeXt', base / 'mednext'])
    for p in candidates:
        if has_mednext_repo(p):
            return p

    clone_dir = WORK_ROOT / 'MedNeXt'
    if not has_mednext_repo(clone_dir):
        print('Cloning official MedNeXt repo:', MEDNEXT_REPO_URL)
        subprocess.check_call(['git', 'clone', '--depth', '1', MEDNEXT_REPO_URL, str(clone_dir)])
    return clone_dir


MEDNEXT_REPO = find_mednext_repo()
if str(MEDNEXT_REPO) not in sys.path:
    sys.path.insert(0, str(MEDNEXT_REPO))

# Install editable without deps because we only use the architecture. This avoids pulling nnU-Net-v1 deps unnecessarily.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(MEDNEXT_REPO), '--no-deps'])

from nnunet_mednext import create_mednext_v1
print('Official MedNeXt repo:', MEDNEXT_REPO)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


Cloning official MedNeXt repo: https://github.com/MIC-DKFZ/MedNeXt.git


Cloning into '/kaggle/working/MedNeXt'...


Obtaining file:///kaggle/working/MedNeXt
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py develop for mednextv1
Official MedNeXt repo: /kaggle/working/MedNeXt
torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4


In [3]:
def strip_nii_suffix(path_like):
    name = Path(str(path_like)).name
    if name.endswith('.nii.gz'):
        return name[:-7]
    if name.endswith('.nii'):
        return name[:-4]
    return Path(name).stem


def scan_id_from_name(path_like):
    stem = strip_nii_suffix(path_like)
    m = re.search(r'(\d{4})$', stem)
    if not m:
        m = re.search(r'(\d+)$', stem)
    if not m:
        raise ValueError(f'Cannot parse scan id from {path_like}')
    return m.group(1).zfill(4)


def has_nifti_pair_root(path_like):
    p = Path(path_like)
    if not (p / 'set').exists() or not (p / 'label').exists():
        return False
    image_count = len(list((p / 'set').glob('*.nii'))) + len(list((p / 'set').glob('*.nii.gz')))
    mask_count = len(list((p / 'label').glob('*.nii'))) + len(list((p / 'label').glob('*.nii.gz')))
    return image_count > 0 and mask_count > 0


def find_phe_root():
    if USER_PHE_ROOT is not None:
        p = Path(USER_PHE_ROOT)
        if has_nifti_pair_root(p):
            return p
        raise FileNotFoundError(f'USER_PHE_ROOT does not contain set/label NIfTI pairs: {p}')

    candidates = [
        LOCAL_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
        LOCAL_ROOT / 'SubdatasetA_NIFIT' / 'NIFIT',
    ]
    if IS_KAGGLE:
        for base in KAGGLE_INPUT.iterdir():
            candidates.extend([
                base / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'NIFIT',
                base,
            ])
    for p in candidates:
        if has_nifti_pair_root(p):
            return p

    search_roots = [KAGGLE_INPUT] if IS_KAGGLE else [LOCAL_ROOT]
    found = []
    for root in search_roots:
        for set_dir in root.rglob('set'):
            p = set_dir.parent
            if has_nifti_pair_root(p):
                found.append(p)
    if found:
        found = sorted(found, key=lambda p: (('SubdatasetA_NIFIT' not in str(p)), len(str(p))))
        return found[0]
    raise FileNotFoundError('Could not find PHE NIfTI root with set/label. Set USER_PHE_ROOT manually.')


PHE_ROOT = find_phe_root()
PHE_IMAGE_DIR = PHE_ROOT / 'set'
PHE_MASK_DIR = PHE_ROOT / 'label'

image_files = sorted(list(PHE_IMAGE_DIR.glob('*.nii')) + list(PHE_IMAGE_DIR.glob('*.nii.gz')))
mask_files = sorted(list(PHE_MASK_DIR.glob('*.nii')) + list(PHE_MASK_DIR.glob('*.nii.gz')))
mask_by_scan = {scan_id_from_name(p): p for p in mask_files}

rows = []
for image_path in image_files:
    scan_id = scan_id_from_name(image_path)
    mask_path = mask_by_scan.get(scan_id)
    if mask_path is None:
        continue
    if scan_id in TEST_SCAN_IDS:
        split = 'test'
    elif scan_id in VAL_SCAN_IDS:
        split = 'val'
    else:
        split = 'train'
    rows.append({
        'scan_id': scan_id,
        'case_id': f'phe_{scan_id}',
        'split': split,
        'image_path': str(image_path),
        'label_path': str(mask_path),
    })

manifest_df = pd.DataFrame(rows).sort_values(['split', 'scan_id']).reset_index(drop=True)
manifest_csv = OUTPUT_ROOT / '02_16b_mednext_s_phe_only_manifest.csv'
manifest_df.to_csv(manifest_csv, index=False)

print('PHE root :', PHE_ROOT)
print('Images   :', len(image_files))
print('Masks    :', len(mask_files))
print('Matched  :', len(manifest_df))
print(manifest_df['split'].value_counts().to_dict())
print('Manifest :', manifest_csv)
display(manifest_df.groupby('split').head(3))


PHE root : /kaggle/input/datasets/dovune/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT
Images   : 120
Masks    : 120
Matched  : 120
{'train': 99, 'test': 11, 'val': 10}
Manifest : /kaggle/working/outputs_02_16b_kaggle_mednext_s_phe_only_stronger_baseline/02_16b_mednext_s_phe_only_manifest.csv


,scan_id,case_id,split,image_path,label_path
0,0002,phe_0002,test,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
1,0029,phe_0029,test,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
2,0033,phe_0033,test,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
11,0001,phe_0001,train,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
12,0004,phe_0004,train,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
13,0005,phe_0005,train,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
110,0013,phe_0013,val,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
111,0014,phe_0014,val,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
112,0060,phe_0060,val,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...


In [4]:
def read_nifti(path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)
    return arr, img


def write_nifti_like(arr_zyx, reference_img, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_img = sitk.GetImageFromArray(arr_zyx.astype(np.uint8))
    out_img.CopyInformation(reference_img)
    sitk.WriteImage(out_img, str(out_path))


def normalize_ct(arr, window=CT_WINDOW):
    arr = arr.astype(np.float32)
    lo, hi = window
    arr = np.clip(arr, lo, hi)
    arr = (arr - lo) / max(hi - lo, 1e-6)
    arr = arr * 2.0 - 1.0
    return arr.astype(np.float32)


def crop_with_pad(arr, center_zyx, patch_size, pad_value=0):
    patch_size = np.asarray(patch_size, dtype=np.int64)
    center = np.asarray(center_zyx, dtype=np.int64)
    start = center - patch_size // 2
    end = start + patch_size

    src_start = np.maximum(start, 0)
    src_end = np.minimum(end, np.asarray(arr.shape, dtype=np.int64))
    dst_start = src_start - start
    dst_end = dst_start + (src_end - src_start)

    out = np.full(tuple(patch_size), pad_value, dtype=arr.dtype)
    out[
        dst_start[0]:dst_end[0],
        dst_start[1]:dst_end[1],
        dst_start[2]:dst_end[2],
    ] = arr[
        src_start[0]:src_end[0],
        src_start[1]:src_end[1],
        src_start[2]:src_end[2],
    ]
    return out


def pad_to_at_least(arr, min_shape, pad_value=0):
    min_shape = np.asarray(min_shape, dtype=np.int64)
    shape = np.asarray(arr.shape, dtype=np.int64)
    pad_after = np.maximum(min_shape - shape, 0)
    pads = [(0, int(v)) for v in pad_after]
    if any(v > 0 for v in pad_after):
        arr = np.pad(arr, pads, mode='constant', constant_values=pad_value)
    return arr, tuple(int(v) for v in pad_after)


def dice_binary(pred, ref, eps=1e-8):
    pred = pred.astype(bool)
    ref = ref.astype(bool)
    inter = np.logical_and(pred, ref).sum(dtype=np.float64)
    denom = pred.sum(dtype=np.float64) + ref.sum(dtype=np.float64)
    if denom == 0:
        return 1.0
    return float((2.0 * inter + eps) / (denom + eps))


def metric_binary(pred, ref):
    pred = pred.astype(bool)
    ref = ref.astype(bool)
    tp = int(np.logical_and(pred, ref).sum())
    fp = int(np.logical_and(pred, ~ref).sum())
    fn = int(np.logical_and(~pred, ref).sum())
    tn = int(np.logical_and(~pred, ~ref).sum())
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 1.0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 1.0
    return {
        'Dice': float(dice),
        'FN': fn,
        'FP': fp,
        'IoU': float(iou),
        'TN': tn,
        'TP': tp,
        'n_pred': int(pred.sum()),
        'n_ref': int(ref.sum()),
    }


def remove_small_components(mask, min_size=0):
    if min_size <= 0:
        return mask.astype(np.uint8)
    labeled, n = ndi.label(mask > 0)
    if n == 0:
        return mask.astype(np.uint8)
    counts = np.bincount(labeled.ravel())
    keep = np.zeros_like(counts, dtype=bool)
    keep[np.where(counts >= min_size)[0]] = True
    keep[0] = False
    return keep[labeled].astype(np.uint8)


In [5]:
class VolumeCache:
    def __init__(self, cache=True):
        self.cache = cache
        self._cache = {}

    def get(self, row):
        case_id = row['case_id']
        if self.cache and case_id in self._cache:
            return self._cache[case_id]

        image_arr, image_obj = read_nifti(row['image_path'])
        label_arr, _ = read_nifti(row['label_path'])
        image_arr = normalize_ct(image_arr)
        label_arr = (label_arr > 0).astype(np.uint8)
        fg_voxels = np.argwhere(label_arr > 0)
        item = {
            'case_id': case_id,
            'image': image_arr,
            'label': label_arr,
            'image_obj': image_obj,
            'fg_voxels': fg_voxels,
        }
        if self.cache:
            self._cache[case_id] = item
        return item


def augment_patch(x, y):
    # x, y shape: z, y, x. Keep paired spatial transforms identical.
    for axis in range(3):
        if random.random() < AUG_FLIP_PROB:
            x = np.flip(x, axis=axis)
            y = np.flip(y, axis=axis)

    if AUG_INTENSITY_SCALE > 0:
        scale = 1.0 + random.uniform(-AUG_INTENSITY_SCALE, AUG_INTENSITY_SCALE)
        x = x * scale
    if AUG_INTENSITY_SHIFT > 0:
        shift = random.uniform(-AUG_INTENSITY_SHIFT, AUG_INTENSITY_SHIFT)
        x = x + shift
    if AUG_GAUSSIAN_NOISE_STD > 0 and random.random() < 0.5:
        x = x + np.random.normal(0.0, AUG_GAUSSIAN_NOISE_STD, size=x.shape).astype(np.float32)

    x = np.clip(x, -1.5, 1.5)
    return x, y


class RandomPatchDataset(Dataset):
    def __init__(self, rows_df, cache, patches_per_epoch, patch_size=PATCH_SIZE, fg_patch_prob=FG_PATCH_PROB, augment=True):
        self.rows = rows_df.reset_index(drop=True).to_dict('records')
        self.cache = cache
        self.patches_per_epoch = int(patches_per_epoch)
        self.patch_size = tuple(int(x) for x in patch_size)
        self.fg_patch_prob = float(fg_patch_prob)
        self.augment = bool(augment)

    def __len__(self):
        return self.patches_per_epoch

    def __getitem__(self, idx):
        row = random.choice(self.rows)
        item = self.cache.get(row)
        img = item['image']
        lab = item['label']
        fg = item['fg_voxels']

        if len(fg) > 0 and random.random() < self.fg_patch_prob:
            center = fg[random.randrange(len(fg))]
            jitter = np.array([random.randint(-8, 8), random.randint(-48, 48), random.randint(-48, 48)])
            center = center + jitter
        else:
            center = np.array([random.randrange(s) for s in img.shape])

        x = crop_with_pad(img, center, self.patch_size, pad_value=-1.0)
        y = crop_with_pad(lab, center, self.patch_size, pad_value=0)
        if self.augment:
            x, y = augment_patch(x, y)

        x = np.ascontiguousarray(x[None], dtype=np.float32)
        y = np.ascontiguousarray(y[None], dtype=np.float32)
        return torch.from_numpy(x), torch.from_numpy(y)


train_df = manifest_df[manifest_df['split'] == 'train'].reset_index(drop=True)
val_df = manifest_df[manifest_df['split'] == 'val'].reset_index(drop=True)
test_df = manifest_df[manifest_df['split'] == 'test'].reset_index(drop=True)

cache = VolumeCache(cache=True)
train_ds = RandomPatchDataset(train_df, cache, TRAIN_PATCHES_PER_EPOCH, augment=True)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print('train/val/test:', len(train_df), len(val_df), len(test_df))
print('train patches per epoch:', len(train_ds))
print('augmentation: flip/intensity/noise enabled')


train/val/test: 99 10 11
train patches per epoch: 256
augmentation: flip/intensity/noise enabled


In [6]:
def build_mednext_model():
    model = create_mednext_v1(
        num_input_channels=1,
        num_classes=1,
        model_id=MEDNEXT_MODEL_ID,
        kernel_size=MEDNEXT_KERNEL_SIZE,
        deep_supervision=MEDNEXT_DEEP_SUPERVISION,
    )
    return model


model = build_mednext_model()
num_params = sum(p.numel() for p in model.parameters())
print(f'Official MedNeXt-{MEDNEXT_MODEL_ID} params:', f'{num_params/1e6:.2f}M')
print('kernel_size:', MEDNEXT_KERNEL_SIZE, 'deep_supervision:', MEDNEXT_DEEP_SUPERVISION)


Official MedNeXt-S params: 5.55M
kernel_size: 3 deep_supervision: False


In [7]:
def primary_logits(output):
    if isinstance(output, (list, tuple)):
        return output[0]
    return output


def sigmoid_np(logits):
    logits = np.clip(logits, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-logits))


def dice_loss_with_logits(logits, targets, eps=1e-5):
    probs = torch.sigmoid(logits)
    dims = tuple(range(1, probs.ndim))
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)
    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def focal_bce_with_logits(logits, targets, alpha=0.75, gamma=2.0):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    probs = torch.sigmoid(logits)
    p_t = probs * targets + (1.0 - probs) * (1.0 - targets)
    alpha_t = alpha * targets + (1.0 - alpha) * (1.0 - targets)
    loss = alpha_t * torch.pow(1.0 - p_t, gamma) * bce
    return loss.mean()


def combined_loss(logits, targets):
    dl = dice_loss_with_logits(logits, targets)
    fl = focal_bce_with_logits(logits, targets)
    return 0.65 * dl + 0.35 * fl


def iter_starts(size, patch, overlap):
    if size <= patch:
        return [0]
    step = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, step))
    last = size - patch
    if starts[-1] != last:
        starts.append(last)
    return starts


@torch.no_grad()
def sliding_window_predict_logits(model, image_zyx, device, patch_size=PATCH_SIZE, overlap=SLIDING_WINDOW_OVERLAP):
    model.eval()
    original_shape = image_zyx.shape
    image_pad, pad_after = pad_to_at_least(image_zyx, patch_size, pad_value=-1.0)
    zdim, ydim, xdim = image_pad.shape
    pz, py, px = patch_size
    z_starts = iter_starts(zdim, pz, overlap)
    y_starts = iter_starts(ydim, py, overlap)
    x_starts = iter_starts(xdim, px, overlap)

    logits_sum = np.zeros(image_pad.shape, dtype=np.float32)
    count = np.zeros(image_pad.shape, dtype=np.float32)

    for z in z_starts:
        for y in y_starts:
            for x in x_starts:
                patch = image_pad[z:z+pz, y:y+py, x:x+px]
                inp = torch.from_numpy(patch[None, None].astype(np.float32)).to(device)
                with autocast(enabled=(USE_AMP and device.type == 'cuda')):
                    logits = primary_logits(model(inp))
                logits_np = logits.detach().float().cpu().numpy()[0, 0]
                logits_sum[z:z+pz, y:y+py, x:x+px] += logits_np
                count[z:z+pz, y:y+py, x:x+px] += 1.0

    logits_avg = logits_sum / np.maximum(count, 1e-6)
    oz, oy, ox = original_shape
    return logits_avg[:oz, :oy, :ox]


@torch.no_grad()
def collect_predictions(model, rows_df, device):
    preds = []
    for row in rows_df.reset_index(drop=True).to_dict('records'):
        item = cache.get(row)
        logits = sliding_window_predict_logits(model, item['image'], device)
        prob = sigmoid_np(logits).astype(np.float32)
        preds.append({'case_id': row['case_id'], 'prob': prob, 'label': item['label']})
    return preds


def score_prediction_cache(preds, threshold, min_component_size=0):
    rows = []
    for item in preds:
        pred = (item['prob'] >= threshold).astype(np.uint8)
        pred = remove_small_components(pred, min_component_size)
        metrics = metric_binary(pred, item['label'])
        rows.append({'case_id': item['case_id'], **metrics})
    df = pd.DataFrame(rows)
    return float(df['Dice'].mean()), df


def evaluate_rows_tuned(model, rows_df, device):
    preds = collect_predictions(model, rows_df, device)
    best = {'dice': -1.0, 'threshold': THRESHOLD, 'min_component_size': DEFAULT_POSTPROCESS_MIN_COMPONENT_SIZE, 'df': None}
    tune_rows = []
    for min_size in POSTPROCESS_MIN_COMPONENT_SIZE_GRID:
        for thr in THRESHOLD_GRID:
            mean_dice, df = score_prediction_cache(preds, thr, min_size)
            tune_rows.append({'threshold': thr, 'min_component_size': min_size, 'mean_dice': mean_dice})
            if mean_dice > best['dice']:
                best = {'dice': mean_dice, 'threshold': float(thr), 'min_component_size': int(min_size), 'df': df}

    tune_df = pd.DataFrame(tune_rows).sort_values('mean_dice', ascending=False).reset_index(drop=True)
    print('Best val setting:', {'threshold': best['threshold'], 'min_component_size': best['min_component_size'], 'dice': best['dice']})
    display(tune_df.head(8))
    for row in best['df'].sort_values('case_id').itertuples(index=False):
        print(f'val {row.case_id}: Dice {row.Dice:.4f}')
    return best, tune_df


In [8]:
RUN_TRAIN = True
RESUME_CHECKPOINT = None  # path neu muon resume

if RUN_TRAIN:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    scaler = GradScaler(enabled=(USE_AMP and device.type == 'cuda'))

    start_epoch = 1
    best_val_dice = -1.0
    best_threshold = THRESHOLD
    best_min_component_size = DEFAULT_POSTPROCESS_MIN_COMPONENT_SIZE
    history = []

    if RESUME_CHECKPOINT is not None:
        try:
            ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
        except TypeError:
            ckpt = torch.load(RESUME_CHECKPOINT, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = int(ckpt.get('epoch', 0)) + 1
        best_val_dice = float(ckpt.get('best_val_dice', -1.0))
        best_threshold = float(ckpt.get('best_threshold', THRESHOLD))
        best_min_component_size = int(ckpt.get('best_min_component_size', DEFAULT_POSTPROCESS_MIN_COMPONENT_SIZE))
        print('Resumed from', RESUME_CHECKPOINT, 'start_epoch', start_epoch, 'best', best_val_dice)

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        t0 = time.time()
        model.train()
        losses = []
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=(USE_AMP and device.type == 'cuda')):
                logits = primary_logits(model(images))
                loss = combined_loss(logits, labels)
            scaler.scale(loss).backward()
            if GRAD_CLIP_NORM is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        scheduler.step()
        train_loss = float(np.mean(losses)) if losses else np.nan
        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'lr': optimizer.param_groups[0]['lr'],
            'val_dice': np.nan,
            'best_threshold_this_eval': np.nan,
            'best_min_component_size_this_eval': np.nan,
            'epoch_time_sec': time.time() - t0,
        }

        do_val = epoch == 1 or epoch == MAX_EPOCHS or (epoch % VALIDATE_EVERY == 0)
        if do_val:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            best_eval, tune_df = evaluate_rows_tuned(model, val_df, device)
            val_dice = best_eval['dice']
            row['val_dice'] = val_dice
            row['best_threshold_this_eval'] = best_eval['threshold']
            row['best_min_component_size_this_eval'] = best_eval['min_component_size']
            tune_df.to_csv(OUTPUT_ROOT / f'02_16b_threshold_tuning_epoch_{epoch:03d}.csv', index=False)

            if val_dice > best_val_dice:
                best_val_dice = val_dice
                best_threshold = best_eval['threshold']
                best_min_component_size = best_eval['min_component_size']
                best_path = CHECKPOINT_DIR / 'checkpoint_best.pth'
                torch.save({
                    'epoch': epoch,
                    'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(),
                    'best_val_dice': best_val_dice,
                    'best_threshold': best_threshold,
                    'best_min_component_size': best_min_component_size,
                    'config': {
                        'MAX_EPOCHS': MAX_EPOCHS,
                        'PATCH_SIZE': PATCH_SIZE,
                        'TRAIN_PATCHES_PER_EPOCH': TRAIN_PATCHES_PER_EPOCH,
                        'MEDNEXT_REPO': str(MEDNEXT_REPO),
                        'MEDNEXT_MODEL_ID': MEDNEXT_MODEL_ID,
                        'MEDNEXT_KERNEL_SIZE': MEDNEXT_KERNEL_SIZE,
                        'CT_WINDOW': CT_WINDOW,
                    },
                }, best_path)
                print(f'New best checkpoint: {best_path} val_dice={best_val_dice:.4f} thr={best_threshold} min_cc={best_min_component_size}')

        latest_path = CHECKPOINT_DIR / 'checkpoint_latest.pth'
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_dice': best_val_dice,
            'best_threshold': best_threshold,
            'best_min_component_size': best_min_component_size,
        }, latest_path)

        history.append(row)
        pd.DataFrame(history).to_csv(OUTPUT_ROOT / '02_16b_mednext_s_training_history.csv', index=False)
        print(f"Epoch {epoch:03d}/{MAX_EPOCHS} loss={train_loss:.4f} val={row['val_dice']} best={best_val_dice:.4f} thr={best_threshold} min_cc={best_min_component_size} time={row['epoch_time_sec']:.1f}s")

    print('Training done. Best val Dice:', best_val_dice, 'threshold:', best_threshold, 'min_component_size:', best_min_component_size)
else:
    print('Skipped training. Set RUN_TRAIN=True to train.')


/tmp/ipykernel_58/4166482451.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(USE_AMP and device.type == 'cuda'))
/tmp/ipykernel_58/4166482451.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):
/tmp/ipykernel_58/1826958041.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):


Best val setting: {'threshold': 0.4, 'min_component_size': 100, 'dice': 0.03595564996221708}


,threshold,min_component_size,mean_dice
0,0.40,100,0.035956
1,0.40,50,0.035057
2,0.35,100,0.034493
3,0.45,50,0.034451
4,0.40,10,0.034288
5,0.35,50,0.034206
6,0.35,10,0.033789
7,0.45,10,0.033613


val phe_0013: Dice 0.1243
val phe_0014: Dice 0.0033
val phe_0060: Dice 0.0000
val phe_0078: Dice 0.0717
val phe_0080: Dice 0.0000
val phe_0115: Dice 0.0262
val phe_0119: Dice 0.0000
val phe_0130: Dice 0.0366
val phe_0138: Dice 0.0000
val phe_0160: Dice 0.0975
New best checkpoint: /kaggle/working/outputs_02_16b_kaggle_mednext_s_phe_only_stronger_baseline/checkpoints/checkpoint_best.pth val_dice=0.0360 thr=0.4 min_cc=100
Epoch 001/160 loss=0.6277 val=0.03595564996221708 best=0.0360 thr=0.4 min_cc=100 time=102.6s


/tmp/ipykernel_58/4166482451.py:39: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):


...

Epoch 151/160 loss=0.3914 val=nan best=0.3838 thr=0.6 min_cc=100 time=53.0s
Epoch 152/160 loss=0.3924 val=nan best=0.3838 thr=0.6 min_cc=100 time=52.2s
Epoch 153/160 loss=0.3929 val=nan best=0.3838 thr=0.6 min_cc=100 time=52.5s
Epoch 154/160 loss=0.3752 val=nan best=0.3838 thr=0.6 min_cc=100 time=52.3s
Epoch 155/160 loss=0.3863 val=nan best=0.3838 thr=0.6 min_cc=100 time=52.0s
Epoch 156/160 loss=0.3842 val=nan best=0.3838 thr=0.6 min_cc=100 time=51.9s
Epoch 157/160 loss=0.3761 val=nan best=0.3838 thr=0.6 min_cc=100 time=52.2s
Epoch 158/160 loss=0.3647 val=nan best=0.3838 thr=0.6 min_cc=100 time=52.3s
Epoch 159/160 loss=0.3979 val=nan best=0.3838 thr=0.6 min_cc=100 time=52.2s


/tmp/ipykernel_58/1826958041.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):


Best val setting: {'threshold': 0.6, 'min_component_size': 100, 'dice': 0.3831208482141363}


,threshold,min_component_size,mean_dice
0,0.60,100,0.383121
1,0.75,100,0.381355
2,0.75,50,0.380651
3,0.65,100,0.380350
4,0.70,50,0.380129
5,0.55,100,0.380090
6,0.65,50,0.379891
7,0.70,100,0.379778


val phe_0013: Dice 0.4131
val phe_0014: Dice 0.5048
val phe_0060: Dice 0.4446
val phe_0078: Dice 0.4875
val phe_0080: Dice 0.6674
val phe_0115: Dice 0.2308
val phe_0119: Dice 0.3100
val phe_0130: Dice 0.3349
val phe_0138: Dice 0.2478
val phe_0160: Dice 0.1904
Epoch 160/160 loss=0.3682 val=0.3831208482141363 best=0.3838 thr=0.6 min_cc=100 time=52.4s
Training done. Best val Dice: 0.38382034985625524 threshold: 0.6 min_component_size: 100


In [9]:
RUN_TEST_INFERENCE = True
CHECKPOINT_FOR_TEST = CHECKPOINT_DIR / 'checkpoint_best.pth'


def export_test_metrics(model, rows_df, device, threshold, min_component_size, tag):
    metric_rows = []
    pred_dir = PRED_DIR / tag
    pred_dir.mkdir(parents=True, exist_ok=True)
    for row in rows_df.to_dict('records'):
        print(f'Predicting {row["case_id"]} [{tag}]')
        item = cache.get(row)
        logits = sliding_window_predict_logits(model, item['image'], device)
        prob = sigmoid_np(logits)
        pred = (prob >= threshold).astype(np.uint8)
        pred = remove_small_components(pred, min_component_size)

        out_path = pred_dir / f"{row['case_id']}.nii.gz"
        write_nifti_like(pred, item['image_obj'], out_path)
        metrics = metric_binary(pred, item['label'])
        metric_rows.append({
            'case_id': row['case_id'],
            'prediction_file': str(out_path),
            'reference_file': row['label_path'],
            'threshold': threshold,
            'min_component_size': min_component_size,
            **metrics,
        })
        print(row['case_id'], 'Dice', f"{metrics['Dice']:.4f}", 'IoU', f"{metrics['IoU']:.4f}")

    metrics_df = pd.DataFrame(metric_rows)
    per_case_csv = OUTPUT_ROOT / f'02_16b_mednext_s_phe_only_metrics_per_case_{tag}.csv'
    metrics_df.to_csv(per_case_csv, index=False)

    summary_rows = []
    numeric_cols = [c for c in metrics_df.columns if c not in {'case_id', 'prediction_file', 'reference_file'}]
    for col in numeric_cols:
        vals = pd.to_numeric(metrics_df[col], errors='coerce').dropna()
        if len(vals) == 0:
            continue
        summary_rows.append({
            'label': 'PHE',
            'metric': col,
            'mean': float(vals.mean()),
            'median': float(vals.median()),
            'std': float(vals.std(ddof=0)),
            'n': int(len(vals)),
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_csv = OUTPUT_ROOT / f'02_16b_mednext_s_phe_only_metrics_summary_{tag}.csv'
    summary_df.to_csv(summary_csv, index=False)

    print('Wrote:', per_case_csv)
    print('Wrote:', summary_csv)
    display(summary_df)
    return metrics_df, summary_df


if RUN_TEST_INFERENCE:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_mednext_model().to(device)
    try:
        ckpt = torch.load(CHECKPOINT_FOR_TEST, map_location=device, weights_only=False)
    except TypeError:
        ckpt = torch.load(CHECKPOINT_FOR_TEST, map_location=device)
    model.load_state_dict(ckpt['model'])
    tuned_threshold = float(ckpt.get('best_threshold', THRESHOLD))
    tuned_min_cc = int(ckpt.get('best_min_component_size', DEFAULT_POSTPROCESS_MIN_COMPONENT_SIZE))
    print('Loaded checkpoint:', CHECKPOINT_FOR_TEST)
    print('Checkpoint epoch:', ckpt.get('epoch'), 'best_val_dice:', ckpt.get('best_val_dice'))
    print('Using tuned threshold:', tuned_threshold, 'min_component_size:', tuned_min_cc)

    tuned_metrics_df, tuned_summary_df = export_test_metrics(
        model, test_df, device,
        threshold=tuned_threshold,
        min_component_size=tuned_min_cc,
        tag='tuned',
    )

    fixed_metrics_df, fixed_summary_df = export_test_metrics(
        model, test_df, device,
        threshold=THRESHOLD,
        min_component_size=DEFAULT_POSTPROCESS_MIN_COMPONENT_SIZE,
        tag='fixed_thr_0p5',
    )
else:
    print('Skipped test inference. Set RUN_TEST_INFERENCE=True after training.')


Loaded checkpoint: /kaggle/working/outputs_02_16b_kaggle_mednext_s_phe_only_stronger_baseline/checkpoints/checkpoint_best.pth
Checkpoint epoch: 150 best_val_dice: 0.38382034985625524
Using tuned threshold: 0.6 min_component_size: 100
Predicting phe_0002 [tuned]


/tmp/ipykernel_58/1826958041.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):


phe_0002 Dice 0.5973 IoU 0.4258
Predicting phe_0029 [tuned]
phe_0029 Dice 0.4267 IoU 0.2712
Predicting phe_0033 [tuned]
phe_0033 Dice 0.4399 IoU 0.2820
Predicting phe_0051 [tuned]
phe_0051 Dice 0.2775 IoU 0.1611
Predicting phe_0061 [tuned]
phe_0061 Dice 0.1756 IoU 0.0963
Predicting phe_0084 [tuned]
phe_0084 Dice 0.5691 IoU 0.3977
Predicting phe_0095 [tuned]
phe_0095 Dice 0.4560 IoU 0.2953
Predicting phe_0096 [tuned]
phe_0096 Dice 0.6419 IoU 0.4726
Predicting phe_0113 [tuned]
phe_0113 Dice 0.1588 IoU 0.0862
Predicting phe_0116 [tuned]
phe_0116 Dice 0.2471 IoU 0.1409
Predicting phe_0167 [tuned]
phe_0167 Dice 0.0000 IoU 0.0000
Wrote: /kaggle/working/outputs_02_16b_kaggle_mednext_s_phe_only_stronger_baseline/02_16b_mednext_s_phe_only_metrics_per_case_tuned.csv
Wrote: /kaggle/working/outputs_02_16b_kaggle_mednext_s_phe_only_stronger_baseline/02_16b_mednext_s_phe_only_metrics_summary_tuned.csv


,label,metric,mean,median,std,n
0,PHE,threshold,6.000000e-01,6.000000e-01,1.110223e-16,11
1,PHE,min_component_size,1.000000e+02,1.000000e+02,0.000000e+00,11
2,PHE,Dice,3.627073e-01,4.266988e-01,1.961120e-01,11
3,PHE,FN,1.124818e+03,1.032000e+03,1.081892e+03,11
4,PHE,FP,2.174727e+03,2.014000e+03,1.904201e+03,11
5,PHE,IoU,2.390145e-01,2.712124e-01,1.468498e-01,11
6,PHE,TN,7.835862e+06,8.382707e+06,8.006272e+05,11
7,PHE,TP,1.327182e+03,9.150000e+02,1.390288e+03,11
8,PHE,n_pred,3.501909e+03,2.267000e+03,2.915974e+03,11
9,PHE,n_ref,2.452000e+03,1.635000e+03,2.110822e+03,11


Predicting phe_0002 [fixed_thr_0p5]


/tmp/ipykernel_58/1826958041.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):


phe_0002 Dice 0.5516 IoU 0.3809
Predicting phe_0029 [fixed_thr_0p5]
phe_0029 Dice 0.4078 IoU 0.2562
Predicting phe_0033 [fixed_thr_0p5]
phe_0033 Dice 0.4541 IoU 0.2937
Predicting phe_0051 [fixed_thr_0p5]
phe_0051 Dice 0.2468 IoU 0.1408
Predicting phe_0061 [fixed_thr_0p5]
phe_0061 Dice 0.1583 IoU 0.0860
Predicting phe_0084 [fixed_thr_0p5]
phe_0084 Dice 0.5579 IoU 0.3868
Predicting phe_0095 [fixed_thr_0p5]
phe_0095 Dice 0.4559 IoU 0.2953
Predicting phe_0096 [fixed_thr_0p5]
phe_0096 Dice 0.6303 IoU 0.4601
Predicting phe_0113 [fixed_thr_0p5]
phe_0113 Dice 0.1538 IoU 0.0833
Predicting phe_0116 [fixed_thr_0p5]
phe_0116 Dice 0.2371 IoU 0.1345
Predicting phe_0167 [fixed_thr_0p5]
phe_0167 Dice 0.0000 IoU 0.0000
Wrote: /kaggle/working/outputs_02_16b_kaggle_mednext_s_phe_only_stronger_baseline/02_16b_mednext_s_phe_only_metrics_per_case_fixed_thr_0p5.csv
Wrote: /kaggle/working/outputs_02_16b_kaggle_mednext_s_phe_only_stronger_baseline/02_16b_mednext_s_phe_only_metrics_summary_fixed_thr_0p5.csv


,label,metric,mean,median,std,n
0,PHE,threshold,5.000000e-01,5.000000e-01,0.000000,11
1,PHE,min_component_size,0.000000e+00,0.000000e+00,0.000000,11
2,PHE,Dice,3.503345e-01,4.078466e-01,0.192814,11
3,PHE,FN,1.077364e+03,9.890000e+02,1034.910472,11
4,PHE,FP,2.541818e+03,2.194000e+03,2069.879128,11
5,PHE,IoU,2.288699e-01,2.561604e-01,0.141922,11
6,PHE,TN,7.835495e+06,8.382395e+06,800615.601067,11
7,PHE,TP,1.374636e+03,9.320000e+02,1421.480999,11
8,PHE,n_pred,3.916455e+03,2.578000e+03,3091.631089,11
9,PHE,n_ref,2.452000e+03,1.635000e+03,2110.821512,11


## Compare note

So sanh truc tiep voi cac run cu tren cung test split 11 case:

- `02_15`: nnU-Net CT-only, 60 epoch.
- `02_12b`: nnU-Net CT-only, 120 epoch.
- `02_12`: nnU-Net CT-only, 250 epoch.
- `02_14`: nnU-Net CT + ICH teacher prior, 250 epoch.
- `02_16`: official MedNeXt-S CT-only, custom light training, 120 epoch.
- `02_16b`: official MedNeXt-S CT-only, stronger custom training, tuned threshold/postprocess.

Neu `02_16b` van thap hon `02_12b`, khong nen ket luan MedNeXt-S do; chi nen ket luan custom MedNeXt training pipeline hien tai chua bat kip nnU-Net pipeline.
